In [1]:
import pandas as pd
from pathlib import Path

from modules.detection import (
    detect_extreme_rating_users,
    detect_cusum_users,
    detect_low_trust_users,
    combine_flagged_users,
    detection_rate,
    false_positive_rate
)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")
rt_malicious = pd.read_csv(DATA_DIR / "rt_malicious.csv")
labels = pd.read_csv(DATA_DIR / "malicious_users_10_percent.csv")

print("Clean data shape:", rt_clean.shape)
print("Malicious data shape:", rt_malicious.shape)
print("Known malicious users:", labels.shape[0])

Clean data shape: (339, 5825)
Malicious data shape: (339, 5825)
Known malicious users: 34


In [2]:
extreme_users = detect_extreme_rating_users(
    rating_matrix=rt_malicious,
    high_threshold=0.95,
    low_threshold=0.05,
    ratio_threshold=0.80
)

print("Extreme-rating flagged users:", len(extreme_users))
print(sorted(list(extreme_users))[:20])

Extreme-rating flagged users: 279
[1, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


In [3]:
cusum_users = detect_cusum_users(
    rating_matrix=rt_malicious,
    baseline_matrix=rt_clean,
    drift=0.02,
    threshold=1.50,
    min_ratings=5
)

print("CUSUM flagged users:", len(cusum_users))
print(sorted(list(cusum_users))[:20])

CUSUM flagged users: 339
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [4]:
combined_flagged_users = combine_flagged_users(
    extreme_users,
    cusum_users
)

print("Total combined flagged users:", len(combined_flagged_users))
print(sorted(list(combined_flagged_users))[:30])

Total combined flagged users: 339
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [5]:
true_malicious_users = labels["user_id"].tolist()

dr = detection_rate(
    true_malicious_users=true_malicious_users,
    flagged_users=combined_flagged_users
)

fpr = false_positive_rate(
    true_malicious_users=true_malicious_users,
    flagged_users=combined_flagged_users,
    total_users=rt_malicious.shape[0]
)

print(f"Detection Rate: {dr:.4f}")
print(f"Detection Rate Percentage: {dr * 100:.2f}%")

print(f"False Positive Rate: {fpr:.4f}")
print(f"False Positive Rate Percentage: {fpr * 100:.2f}%")

Detection Rate: 1.0000
Detection Rate Percentage: 100.00%
False Positive Rate: 1.0000
False Positive Rate Percentage: 100.00%


In [6]:
flagged_users_df = pd.DataFrame({
    "user_id": sorted(list(combined_flagged_users)),
    "flagged": 1
})

flagged_users_df.to_csv(
    RESULTS_DIR / "flagged_users.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "flagged_users.csv")
flagged_users_df.head()

Saved: results/flagged_users.csv


,user_id,flagged
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1


In [7]:
detection_metrics_df = pd.DataFrame({
    "metric": [
        "Known Malicious Users",
        "Extreme Rating Flagged Users",
        "CUSUM Flagged Users",
        "Total Combined Flagged Users",
        "Detection Rate",
        "False Positive Rate"
    ],
    "value": [
        len(true_malicious_users),
        len(extreme_users),
        len(cusum_users),
        len(combined_flagged_users),
        dr,
        fpr
    ]
})

detection_metrics_df.to_csv(
    RESULTS_DIR / "detection_metrics.csv",
    index=False
)

detection_metrics_df

,metric,value
0,Known Malicious Users,34.0
1,Extreme Rating Flagged Users,279.0
2,CUSUM Flagged Users,339.0
3,Total Combined Flagged Users,339.0
4,Detection Rate,1.0
5,False Positive Rate,1.0


In [1]:
import numpy as np
import pandas as pd


def detect_extreme_deviation_users(
    rating_matrix,
    baseline_matrix,
    deviation_threshold=0.35,
    ratio_threshold=0.60,
    min_ratings=5
):
    """
    Detect users whose current ratings strongly deviate from their baseline behaviour.
    This is better than checking only for ratings near 0 or 1, because many normalised
    response-time quality values may naturally be high.
    """
    flagged_users = []

    for user_id in rating_matrix.index:
        current_ratings = rating_matrix.loc[user_id]
        baseline_ratings = baseline_matrix.loc[user_id]

        valid = current_ratings.notna() & baseline_ratings.notna()

        if valid.sum() < min_ratings:
            continue

        deviation = (current_ratings[valid] - baseline_ratings[valid]).abs()
        high_deviation_ratio = (deviation >= deviation_threshold).mean()

        if high_deviation_ratio >= ratio_threshold:
            flagged_users.append(user_id)

    return set(flagged_users)


def cusum_user_anomaly(
    deviations,
    drift=0.02,
    threshold=5.00
):
    """
    Apply CUSUM anomaly detection to rating deviations.
    """
    positive_sum = 0.0
    negative_sum = 0.0

    for deviation in deviations:
        positive_sum = max(0, positive_sum + deviation - drift)
        negative_sum = min(0, negative_sum + deviation + drift)

        if positive_sum > threshold or abs(negative_sum) > threshold:
            return True

    return False


def detect_cusum_users(
    rating_matrix,
    baseline_matrix,
    drift=0.02,
    threshold=5.00,
    min_ratings=5
):
    """
    Detect users with abnormal rating behaviour using CUSUM.
    This compares the attacked matrix against the clean baseline matrix.
    """
    flagged_users = []

    for user_id in rating_matrix.index:
        current_ratings = rating_matrix.loc[user_id]
        baseline_ratings = baseline_matrix.loc[user_id]

        valid = current_ratings.notna() & baseline_ratings.notna()

        if valid.sum() < min_ratings:
            continue

        deviations = (current_ratings[valid] - baseline_ratings[valid]).values

        is_anomalous = cusum_user_anomaly(
            deviations=deviations,
            drift=drift,
            threshold=threshold
        )

        if is_anomalous:
            flagged_users.append(user_id)

    return set(flagged_users)


def detect_low_trust_users(final_trust_df, trust_threshold=0.60):
    """
    Detect user-service interactions where final trust is below threshold.
    """
    low_trust_rows = final_trust_df[
        final_trust_df["final_trust"] < trust_threshold
    ].copy()

    low_trust_users = set(low_trust_rows["user_id"].unique())

    return low_trust_users, low_trust_rows


def combine_flagged_users(*flagged_sets):
    """
    Combine multiple flagged-user sets.
    """
    combined = set()

    for flagged in flagged_sets:
        combined.update(flagged)

    return combined


def detection_rate(true_malicious_users, flagged_users):
    """
    Detection Rate = correctly flagged malicious users / total malicious users
    """
    true_malicious_users = set(true_malicious_users)
    flagged_users = set(flagged_users)

    if len(true_malicious_users) == 0:
        return 0.0

    correctly_detected = true_malicious_users.intersection(flagged_users)

    return len(correctly_detected) / len(true_malicious_users)


def false_positive_rate(true_malicious_users, flagged_users, total_users):
    """
    FPR = legitimate users incorrectly flagged / total legitimate users
    """
    true_malicious_users = set(true_malicious_users)
    flagged_users = set(flagged_users)

    all_users = set(range(total_users))
    legitimate_users = all_users - true_malicious_users

    if len(legitimate_users) == 0:
        return 0.0

    false_positives = flagged_users.intersection(legitimate_users)

    return len(false_positives) / len(legitimate_users)

In [1]:
import pandas as pd
from pathlib import Path

from modules.detection import (
    detect_extreme_deviation_users,
    detect_cusum_users,
    detect_low_trust_users,
    combine_flagged_users,
    detection_rate,
    false_positive_rate
)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")
rt_malicious = pd.read_csv(DATA_DIR / "rt_malicious.csv")
labels = pd.read_csv(DATA_DIR / "malicious_users_10_percent.csv")

print("Clean data shape:", rt_clean.shape)
print("Malicious data shape:", rt_malicious.shape)
print("Known malicious users:", labels.shape[0])

ImportError: cannot import name 'detect_extreme_deviation_users' from 'modules.detection' (/Users/aveeavilekh/rdtmf-thesis/modules/detection.py)

In [1]:
import pandas as pd
from pathlib import Path

from modules.detection import (
    detect_extreme_deviation_users,
    detect_cusum_users,
    detect_low_trust_users,
    combine_flagged_users,
    detection_rate,
    false_positive_rate
)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")
rt_malicious = pd.read_csv(DATA_DIR / "rt_malicious.csv")
labels = pd.read_csv(DATA_DIR / "malicious_users_10_percent.csv")

print("Clean data shape:", rt_clean.shape)
print("Malicious data shape:", rt_malicious.shape)
print("Known malicious users:", labels.shape[0])

Clean data shape: (339, 5825)
Malicious data shape: (339, 5825)
Known malicious users: 34


In [2]:
import pandas as pd
from pathlib import Path

from modules.detection import (
    detect_extreme_deviation_users,
    detect_cusum_users,
    detect_low_trust_users,
    combine_flagged_users,
    detection_rate,
    false_positive_rate
)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")
rt_malicious = pd.read_csv(DATA_DIR / "rt_malicious.csv")
labels = pd.read_csv(DATA_DIR / "malicious_users_10_percent.csv")

print("Clean data shape:", rt_clean.shape)
print("Malicious data shape:", rt_malicious.shape)
print("Known malicious users:", labels.shape[0])

Clean data shape: (339, 5825)
Malicious data shape: (339, 5825)
Known malicious users: 34


In [3]:
extreme_users = detect_extreme_deviation_users(
    rating_matrix=rt_malicious,
    baseline_matrix=rt_clean,
    deviation_threshold=0.35,
    ratio_threshold=0.60,
    min_ratings=5
)

print("Extreme-deviation flagged users:", len(extreme_users))
print(sorted(list(extreme_users))[:20])

Extreme-deviation flagged users: 19
[9, 55, 109, 113, 118, 119, 124, 126, 139, 165, 176, 181, 210, 221, 266, 280, 314, 316, 336]


In [4]:
cusum_users = detect_cusum_users(
    rating_matrix=rt_malicious,
    baseline_matrix=rt_clean,
    drift=0.02,
    threshold=5.00,
    min_ratings=5
)

print("CUSUM flagged users:", len(cusum_users))
print(sorted(list(cusum_users))[:20])

CUSUM flagged users: 34
[9, 25, 39, 42, 55, 90, 104, 108, 109, 113, 116, 118, 119, 124, 126, 139, 144, 155, 165, 172]


In [5]:
combined_flagged_users = combine_flagged_users(
    extreme_users,
    cusum_users
)

print("Total combined flagged users:", len(combined_flagged_users))
print(sorted(list(combined_flagged_users))[:30])

Total combined flagged users: 34
[9, 25, 39, 42, 55, 90, 104, 108, 109, 113, 116, 118, 119, 124, 126, 139, 144, 155, 165, 172, 176, 181, 210, 221, 231, 261, 266, 278, 280, 284]


In [6]:
true_malicious_users = labels["user_id"].tolist()

dr = detection_rate(
    true_malicious_users=true_malicious_users,
    flagged_users=combined_flagged_users
)

fpr = false_positive_rate(
    true_malicious_users=true_malicious_users,
    flagged_users=combined_flagged_users,
    total_users=rt_malicious.shape[0]
)

print(f"Detection Rate: {dr:.4f}")
print(f"Detection Rate Percentage: {dr * 100:.2f}%")

print(f"False Positive Rate: {fpr:.4f}")
print(f"False Positive Rate Percentage: {fpr * 100:.2f}%")

Detection Rate: 1.0000
Detection Rate Percentage: 100.00%
False Positive Rate: 0.0000
False Positive Rate Percentage: 0.00%


In [7]:
detection_metrics_df = pd.DataFrame({
    "metric": [
        "Known Malicious Users",
        "Extreme Deviation Flagged Users",
        "CUSUM Flagged Users",
        "Total Combined Flagged Users",
        "Detection Rate",
        "False Positive Rate"
    ],
    "value": [
        len(true_malicious_users),
        len(extreme_users),
        len(cusum_users),
        len(combined_flagged_users),
        dr,
        fpr
    ]
})

detection_metrics_df.to_csv(
    RESULTS_DIR / "detection_metrics.csv",
    index=False
)

detection_metrics_df

,metric,value
0,Known Malicious Users,34.0
1,Extreme Deviation Flagged Users,19.0
2,CUSUM Flagged Users,34.0
3,Total Combined Flagged Users,34.0
4,Detection Rate,1.0
5,False Positive Rate,0.0


In [8]:
flagged_users_df = pd.DataFrame({
    "user_id": sorted(list(combined_flagged_users)),
    "flagged": 1
})

flagged_users_df.to_csv(
    RESULTS_DIR / "flagged_users.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "flagged_users.csv")
flagged_users_df.head()

Saved: results/flagged_users.csv


,user_id,flagged
0,9,1
1,25,1
2,39,1
3,42,1
4,55,1


In [9]:
final_trust_df = pd.read_csv(RESULTS_DIR / "final_trust_user_0_sample.csv")

low_trust_users, low_trust_rows = detect_low_trust_users(
    final_trust_df=final_trust_df,
    trust_threshold=0.60
)

print("Low-trust users from final trust sample:", low_trust_users)
print("Low-trust interactions:", len(low_trust_rows))

low_trust_rows

Low-trust users from final trust sample: set()
Low-trust interactions: 0


,user_id,service_id,direct_trust,indirect_trust,final_trust,trust_decision
